This notebook constructs a dataset for modelling next-day sales at the single-product level, using only information available at the end of the current day. The workflow defines the forecasting scope and target, validates the cleaned transactions, aggregates quantities into a daily sales series, and distinguishes zero recorded sales from missing or uncertain observations. Feature engineering then captures historical sales, recent trends and variability, calendar patterns, and relevant transaction context where available. Chronological training, validation, and test periods preserve temporal order, while checks for feature availability, missing values, and data leakage ensure that the resulting dataset is suitable for modelling. Baseline forecasts use the previous day’s sales, sales on the same weekday of the previous week, and a recent rolling average.

Observed sales are treated as a proxy for demand. This approximation assumes that sufficient stock was available and that customers could complete their intended purchases. However, stock levels, stockouts, unmet orders, product availability, and other factors influencing purchasing behaviour are not fully observed. Consequently, low or zero sales cannot always be interpreted as low or zero demand, and the model forecasts realised sales rather than directly estimating unconstrained demand. These limitations and assumptions are documented explicitly so that subsequent stock optimisation can account for uncertainty about demand, inventory availability, replenishment lead times, and relevant costs.

In [1]:
# import relevant libraries
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf
from pathlib import Path


# set path to the raw data file
PROJECT_ROOT = Path.cwd().parent
raw_data_path = PROJECT_ROOT / "data/raw/online_retail_II.csv"
df = pd.read_csv(raw_data_path)

In [2]:
# quick overview about the dataset
print("Number of rows and columns:", df.shape)
print("\n Column names: ")
print(df.columns.tolist())

display(df.head())
display(df.info())

overview = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
    "unique_values": df.nunique(),
})

display(overview)

print("Exact duplicates:", df.duplicated().sum())

Number of rows and columns: (1067371, 8)

 Column names: 
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


None

,data_type,missing_count,missing_percent,unique_values
Invoice,str,0,0.00,53628
StockCode,str,0,0.00,5305
Description,str,4382,0.41,5698
Quantity,int64,0,0.00,1057
InvoiceDate,str,0,0.00,47635
Price,float64,0,0.00,2807
Customer ID,float64,243007,22.77,5942
Country,str,0,0.00,43


Exact duplicates: 34335


In [13]:
df["StockCode"].nunique()

5305

In [3]:
# track unusual codes for product (StockCode)
audit = df.copy()

audit["StockCode"] = audit["StockCode"].astype("string").str.strip()

code_summary = (
    audit.groupby("StockCode", dropna=False)
    .agg(
        descriptions=(
            "Description",
            lambda s: sorted(set(s.dropna().astype(str)))
        ),
        records=("Quantity", "size"),
        min_quantity=("Quantity", "min"),
        max_quantity=("Quantity", "max"),
        min_price=("Price", "min"),
        max_price=("Price", "max"),
        zero_price_share=("Price", lambda s: s.eq(0).mean()),
        missing_customer_share=(
            "Customer ID", lambda s: s.isna().mean()
        ),
    )
)

display(code_summary)

,descriptions,records,min_quantity,max_quantity,min_price,max_price,zero_price_share,missing_customer_share
StockCode,,,,,,,,
10002,[INFLATABLE POLITICAL GLOBE ],400,-400,400,0.00,1.70,0.005000,0.190000
10002R,[ROBOT PENCIL SHARPNER],3,1,2,5.06,5.17,0.000000,1.000000
10080,"[GROOVY CACTUS INFLATABLE, check]",31,1,170,0.00,0.85,0.096774,0.129032
10109,[BENDY COLOUR PENCILS],2,-4,4,0.00,0.42,0.500000,0.500000
10120,"[DOGGY RUBBER, Zebra invcing error]",79,-9000,60,0.00,0.43,0.012658,0.151899
...,...,...,...,...,...,...,...,...
gift_0001_60,[],2,3,10,0.00,0.00,1.000000,1.000000
gift_0001_70,[Dotcomgiftshop Gift Voucher £70.00],3,1,10,0.00,59.57,0.666667,1.000000
gift_0001_80,[Dotcomgiftshop Gift Voucher £80.00],4,-1,10,0.00,69.56,0.500000,1.000000


In [4]:
usual_format = (
    code_summary.index.to_series()
    .str.fullmatch(r"\d{5}[A-Za-z]*", na=False)
)

display(code_summary.loc[~usual_format])

,descriptions,records,min_quantity,max_quantity,min_price,max_price,zero_price_share,missing_customer_share
StockCode,,,,,,,,
ADJUST,"[Adjustment by Peter on 24/05/2010 1, Adjustme...",67,-1,1,4.570,5117.03,0.000000,0.089552
ADJUST2,[Adjustment by Peter on Jun 25 2010 ],3,1,1,72.450,358.47,0.000000,0.000000
AMAZONFEE,[AMAZON FEE],43,-1,1,1.000,17836.46,0.000000,1.000000
B,[Adjust bad debt],6,1,1,-53594.360,11062.06,0.000000,1.000000
BANK CHARGES,"[ Bank Charges, Bank Charges]",102,-11,1,0.001,18910.69,0.000000,0.627451
...,...,...,...,...,...,...,...,...
gift_0001_60,[],2,3,10,0.000,0.00,1.000000,1.000000
gift_0001_70,[Dotcomgiftshop Gift Voucher £70.00],3,1,10,0.000,59.57,0.666667,1.000000
gift_0001_80,[Dotcomgiftshop Gift Voucher £80.00],4,-1,10,0.000,69.56,0.500000,1.000000


In [5]:
# display them entirely for further investigation
unusual_codes = code_summary.loc[~usual_format]

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None,
):
    display(unusual_codes[["descriptions", "records"]])

,descriptions,records
StockCode,,
ADJUST,"[Adjustment by Peter on 24/05/2010 1, Adjustment by john on 26/01/2010 16, Adjustment by john on 26/01/2010 17]",67
ADJUST2,[Adjustment by Peter on Jun 25 2010 ],3
AMAZONFEE,[AMAZON FEE],43
B,[Adjust bad debt],6
BANK CHARGES,"[ Bank Charges, Bank Charges]",102
C2,[CARRIAGE],282
C3,[],1
CRUK,[CRUK Commission],16
D,[Discount],177


In [6]:
# After deciding on which to keep and what to exclude, the product codes for the dataset become

codes = df["StockCode"].astype("string").str.strip().str.upper()

# Categories identified during the audit
excluded_categories = {
    "ADJUST", "ADJUST2", "AMAZONFEE", "B", "BANK CHARGES",
    "C2", "CRUK", "D", "DOT", "M", "POST", "S",
}

exclude_mask = (
    codes.isin(excluded_categories)
    | codes.str.startswith("GIFT_0001_", na=False)
    | codes.str.fullmatch(r"TEST\d+", na=False)
)

# List of excluded codes actually present in the dataset
non_product_codes = sorted(
    codes.loc[exclude_mask].dropna().unique().tolist()
)

print(non_product_codes)

excluded_records = df.loc[exclude_mask].copy()
product_records = df.loc[~exclude_mask].copy()

['ADJUST', 'ADJUST2', 'AMAZONFEE', 'B', 'BANK CHARGES', 'C2', 'CRUK', 'D', 'DOT', 'GIFT_0001_10', 'GIFT_0001_20', 'GIFT_0001_30', 'GIFT_0001_40', 'GIFT_0001_50', 'GIFT_0001_60', 'GIFT_0001_70', 'GIFT_0001_80', 'GIFT_0001_90', 'M', 'POST', 'S', 'TEST001', 'TEST002']


In [7]:
# apply data cleaning steps from EDA notebook
# 1. Drop exact duplicates
candidate_sales = df.drop_duplicates().copy()
# 2. type cast invoice variable to string format and remove any space
invoice = candidate_sales["Invoice"].astype("string").str.strip().str.upper()
# 3. Do the same for StockCode, the product identifier
stock_code = candidate_sales["StockCode"].astype("string").str.strip()
# 4. non-product records, defined above
non_product_codes = non_product_codes
# 5. exclude irrelevant records
exclude = (
    stock_code.isin(non_product_codes)
    | invoice.str.startswith("C", na=False)
    | candidate_sales["Quantity"].lt(0)
)

candidate_sales = candidate_sales.loc[~exclude].copy()

In [8]:
# apply further filters to handle zero price and quantity records, negative price and voided purchases

invoice1 = candidate_sales["Invoice"].astype("string").str.strip()
stock_code1 = candidate_sales["StockCode"].astype("string").str.strip()

known_voided_sales = (
    (invoice1.eq("581483") & stock_code1.eq("23843"))
    | (invoice1.eq("541431") & stock_code1.eq("23166"))
)

clean_df = candidate_sales.loc[
    candidate_sales["Quantity"].gt(0)
    & candidate_sales["Price"].gt(0)
    & ~known_voided_sales
].copy()

In [9]:
# Select one product to start feature engineering
selected_code = "22041"  # Replace with the selected StockCode

product_df = clean_df.loc[
    clean_df["StockCode"].astype("string").eq(selected_code)
].copy()

product_df["InvoiceDate"] = pd.to_datetime(product_df["InvoiceDate"])

if product_df.empty:
    raise ValueError("No sales found for the selected product.")
# define daily sales as the total quantity sold on each date
daily_sales = (
    product_df.set_index("InvoiceDate")["Quantity"]
              .sort_index()
              .resample("D")
              .sum()
)

In [10]:
# Feature set for the selected product: one row per forecast date.
# Target: sales on that date.
# Historical features: information available before that date.

daily_sales = daily_sales.sort_index().rename("units_sold")
daily_sales.index.name = "forecast_date"

features_df = daily_sales.to_frame(name="target_sales")

# 1. Historical sales lags.
for lag in [1, 7, 14, 28]:
    features_df[f"sales_lag_{lag}"] = daily_sales.shift(lag)

# 2. Rolling means and standard deviations.
# Shift first to exclude the forecast day's sales.
past_sales = daily_sales.shift(1)

for window in [7, 28]:
    features_df[f"sales_mean_{window}"] = (
        past_sales.rolling(window, min_periods=window).mean()
    )
    features_df[f"sales_std_{window}"] = (
        past_sales.rolling(window, min_periods=window).std()
    )

# 3. Calendar features, known before the forecast date.
features_df["forecast_weekday"] = features_df.index.dayofweek # Monday = 0
features_df["forecast_month"] = features_df.index.month

# OPTIONAL FEATURES: examine in later experiments.

# Frequency of zero recorded sales.
past_zero_sales = (
    past_sales.eq(0).astype(float).where(past_sales.notna())
)
features_df["zero_sales_days_28"] = (
    past_zero_sales.rolling(28, min_periods=28).sum()
)

# Recent sales level relative to the longer-term average.
features_df["sales_trend_7_vs_28"] = (
    features_df["sales_mean_7"] - features_df["sales_mean_28"]
)

# Historical high-sales indicators.
# Illustrative threshold: justify using business knowledge
# or determine it using training data only.
high_sales_threshold = 50
past_high_sales = (
    past_sales.ge(high_sales_threshold)
    .astype(float)
    .where(past_sales.notna())
)
features_df["high_sales_days_28"] = (
    past_high_sales.rolling(28, min_periods=28).sum()
)
features_df["high_sales_lag_1"] = past_high_sales

# Explicit predictor list for the first experiment.
# Add optional feature names here when evaluating them later.
feature_columns = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_mean_7",
    "sales_std_7",
    "sales_mean_28",
    "sales_std_28",
    "forecast_weekday",
    "forecast_month",
]

# Simple baseline forecasts for comparison.
baseline_columns = {
    "previous_day": "sales_lag_1",
    "same_weekday_last_week": "sales_lag_7",
    "recent_average": "sales_mean_7",
}

# Keep rows with a known target and complete initial features.
# Zero-sales observations remain in the dataset.
model_df = features_df.dropna(
    subset=feature_columns + ["target_sales"]
).copy()

if model_df.empty:
    raise ValueError(
        "No usable rows remain. Check for sufficient history "
        "and missing daily observations."
    )

# Product identifier is metadata, not a predictor.
model_df.insert(0, "StockCode", selected_code)

X = model_df[feature_columns].copy()
y = model_df["target_sales"].copy()

display(model_df.tail(10))
print(
    f"Modelling dataset: {len(model_df)} rows, "
    f"{len(feature_columns)} features"
)

,StockCode,target_sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_mean_7,sales_std_7,sales_mean_28,sales_std_28,forecast_weekday,forecast_month,zero_sales_days_28,sales_trend_7_vs_28,high_sales_days_28,high_sales_lag_1
forecast_date,,,,,,,,,,,,,,,,
2011-11-30,22041,106,9.0,15.0,11.0,0.0,5.285714,5.345225,18.928571,38.299916,2,11,11.0,-13.642857,4.0,0.0
2011-12-01,22041,1,106.0,3.0,3.0,6.0,18.285714,38.810283,22.714286,41.467397,3,12,10.0,-4.428571,5.0,1.0
2011-12-02,22041,2,1.0,6.0,30.0,48.0,18.000000,38.948684,22.535714,41.552717,4,12,10.0,-4.535714,5.0,0.0
2011-12-03,22041,0,2.0,0.0,0.0,0.0,17.428571,39.182722,20.892857,41.417780,5,12,10.0,-3.464286,5.0,0.0
2011-12-04,22041,0,0.0,0.0,0.0,4.0,17.428571,39.182722,20.892857,41.417780,6,12,10.0,-3.464286,5.0,0.0
2011-12-05,22041,19,0.0,4.0,18.0,58.0,17.428571,39.182722,20.750000,41.485049,0,12,11.0,-3.321429,5.0,0.0
2011-12-06,22041,27,19.0,9.0,0.0,6.0,19.571429,38.733522,19.357143,40.837722,1,12,11.0,0.214286,4.0,0.0
2011-12-07,22041,3,27.0,106.0,15.0,62.0,22.142857,38.511594,20.107143,40.776118,2,12,11.0,2.035714,4.0,0.0
2011-12-08,22041,3,3.0,1.0,3.0,61.0,7.428571,10.937049,18.000000,40.049044,3,12,11.0,-10.571429,3.0,0.0


Modelling dataset: 711 rows, 10 features


In [12]:
# Extended feature set: one row per forecast date.
# All historical features use information available before that date.

daily_sales = daily_sales.sort_index().rename("units_sold")
daily_sales.index.name = "forecast_date"

features_df_v2 = daily_sales.to_frame(name="target_sales")

# 1. Historical sales lags.
for lag in [1, 7, 14, 28]:
    features_df_v2[f"sales_lag_{lag}"] = daily_sales.shift(lag)

# 2. Rolling means and standard deviations.
past_sales = daily_sales.shift(1)

for window in [7, 28]:
    features_df_v2[f"sales_mean_{window}"] = (
        past_sales.rolling(window, min_periods=window).mean()
    )
    features_df_v2[f"sales_std_{window}"] = (
        past_sales.rolling(window, min_periods=window).std()
    )

# 3. Calendar features.
features_df_v2["forecast_weekday"] = features_df_v2.index.dayofweek
features_df_v2["forecast_month"] = features_df_v2.index.month

# 4. Zero-sales frequency.
past_zero_sales = (
    past_sales.eq(0)
    .astype(float)
    .where(past_sales.notna())
)

features_df_v2["zero_sales_days_28"] = (
    past_zero_sales.rolling(28, min_periods=28).sum()
)

# 5. Recent sales change.
features_df_v2["sales_trend_7_vs_28"] = (
    features_df_v2["sales_mean_7"]
    - features_df_v2["sales_mean_28"]
)

# 6. Historical high-sales indicators.
# Fixed illustrative threshold, not estimated from future observations.
# Daily totals identify high-sales days, not necessarily bulk orders.
high_sales_threshold = 50

past_high_sales = (
    past_sales.ge(high_sales_threshold)
    .astype(float)
    .where(past_sales.notna())
)

features_df_v2["high_sales_days_28"] = (
    past_high_sales.rolling(28, min_periods=28).sum()
)
features_df_v2["high_sales_lag_1"] = past_high_sales

# Original predictors.
initial_features = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_mean_7",
    "sales_std_7",
    "sales_mean_28",
    "sales_std_28",
    "forecast_weekday",
    "forecast_month",
]

# Additional groups allow separate and combined experiments.
additional_feature_groups = {
    "zero_sales": ["zero_sales_days_28"],
    "change": ["sales_trend_7_vs_28"],
    "high_sales": ["high_sales_days_28", "high_sales_lag_1"],
}

new_features = [
    feature
    for group in additional_feature_groups.values()
    for feature in group
]

feature_columns_v2 = initial_features + new_features

# Baselines remain unchanged.
baseline_columns = {
    "previous_day": "sales_lag_1",
    "same_weekday_last_week": "sales_lag_7",
    "recent_average": "sales_mean_7",
}

# Retain known targets and complete predictors, including zero-sales days.
model_df_v2 = features_df_v2.dropna(
    subset=feature_columns_v2 + ["target_sales"]
).copy()

if model_df_v2.empty:
    raise ValueError(
        "No usable rows remain. Check for sufficient history "
        "and missing daily observations."
    )

# Product identifier is metadata.
model_df_v2.insert(0, "StockCode", selected_code)

X_v2 = model_df_v2[feature_columns_v2].copy()
y_v2 = model_df_v2["target_sales"].copy()

display(model_df_v2.tail(10))
print(
    f"Extended modelling dataset: {len(model_df_v2)} rows, "
    f"{len(feature_columns_v2)} features"
)

,StockCode,target_sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_mean_7,sales_std_7,sales_mean_28,sales_std_28,forecast_weekday,forecast_month,zero_sales_days_28,sales_trend_7_vs_28,high_sales_days_28,high_sales_lag_1
forecast_date,,,,,,,,,,,,,,,,
2011-11-30,22041,106,9.0,15.0,11.0,0.0,5.285714,5.345225,18.928571,38.299916,2,11,11.0,-13.642857,4.0,0.0
2011-12-01,22041,1,106.0,3.0,3.0,6.0,18.285714,38.810283,22.714286,41.467397,3,12,10.0,-4.428571,5.0,1.0
2011-12-02,22041,2,1.0,6.0,30.0,48.0,18.000000,38.948684,22.535714,41.552717,4,12,10.0,-4.535714,5.0,0.0
2011-12-03,22041,0,2.0,0.0,0.0,0.0,17.428571,39.182722,20.892857,41.417780,5,12,10.0,-3.464286,5.0,0.0
2011-12-04,22041,0,0.0,0.0,0.0,4.0,17.428571,39.182722,20.892857,41.417780,6,12,10.0,-3.464286,5.0,0.0
2011-12-05,22041,19,0.0,4.0,18.0,58.0,17.428571,39.182722,20.750000,41.485049,0,12,11.0,-3.321429,5.0,0.0
2011-12-06,22041,27,19.0,9.0,0.0,6.0,19.571429,38.733522,19.357143,40.837722,1,12,11.0,0.214286,4.0,0.0
2011-12-07,22041,3,27.0,106.0,15.0,62.0,22.142857,38.511594,20.107143,40.776118,2,12,11.0,2.035714,4.0,0.0
2011-12-08,22041,3,3.0,1.0,3.0,61.0,7.428571,10.937049,18.000000,40.049044,3,12,11.0,-10.571429,3.0,0.0


Extended modelling dataset: 711 rows, 14 features


## Features selected, Assumptions made and Cleaning rules 

The modelling dataset is saved with one row per forecast date, including the product identifier, daily sales target, and ten initial features: historical sales lags, rolling means and standard deviations, weekday, and month. Before saving, daily coverage, incomplete boundary days, feature alignment, and missing values are checked. Zero recorded sales are retained, while observed sales remain a proxy for demand under the documented assumptions about stock availability and other unobserved factors. Chronological training, validation, and test boundaries will be defined before modelling. Optional features will be evaluated in later experiments.

In [ ]:
# Save the dataset after basic consistency checks

assert model_df.index.is_unique, "Duplicate forecast dates found."
assert model_df.index.is_monotonic_increasing, "Dates are not sorted."
assert model_df[feature_columns + ["target_sales"]].notna().all().all(), (
    "Missing feature or target values found."
)
assert model_df["target_sales"].ge(0).all(), "Negative sales targets found."

# Save the forecast date as an explicit column.
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"product_{selected_code}_features_v1.csv"

model_df.reset_index().to_csv(output_path, index=False)

print(f"Saved {len(model_df)} rows to {output_path}")

Saved 739 rows to c:\Users\patri\smart-stock\data\processed\product_22041_features_v2.csv


In [ ]:
# Save the dataset of extended features after basic consistency checks

assert model_df_v2.index.is_unique, "Duplicate forecast dates found."
assert model_df_v2.index.is_monotonic_increasing, "Dates are not sorted."
assert model_df_v2[feature_columns + ["target_sales"]].notna().all().all(), (
    "Missing feature or target values found."
)
assert model_df_v2["target_sales"].ge(0).all(), "Negative sales targets found."

# Save the forecast date as an explicit column.
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path2 = output_dir / f"product_{selected_code}_features_v2.csv"

model_df_v2.reset_index().to_csv(output_path2, index=False)

print(f"Saved {len(model_df_v2)} rows to {output_path2}")

Saved 711 rows to c:\Users\patri\smart-stock\data\processed\product_22041_features_v2.csv
